# 02 — Model Families and Modality Interfaces

> **Orbit 5 (Multimodal)** · **Difficulty**: Intermediate · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/02_model_families_and_modality_interfaces.ipynb)

**Prerequisites**: [01_intro_to_multimodal_systems.ipynb](01_intro_to_multimodal_systems.ipynb)

**What you will learn**:
- The 4 families of multimodal models and how each processes inputs
- How to load, configure, and call each model type on Colab Pro
- The common pattern: processor → model → structured output
- When to use which family for which task

In [ ]:
# --- Setup: Load VLM ---
!pip install -q "transformers>=4.57.0" accelerate bitsandbytes torch numpy pandas matplotlib Pillow qwen-vl-utils

import json, math, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor, BitsAndBytesConfig

plt.style.use("seaborn-v0_8-whitegrid")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/models"
except Exception:
    CACHE_DIR = "./models"

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)
print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## 1 — The Four Model Families

Each family processes different modalities through specialized pipelines:

| Family | Input | Processing | Output | Key models |
|--------|-------|-----------|--------|------------|
| **VLMs** | Image + text prompt | Image → patches → cross-attention with text | Text (answers, descriptions) | Qwen2-VL, InternVL2, LLaVA |
| **Document** | Page image | Page → patch embeddings (with layout awareness) | Retrieval scores, extracted fields | ColPali, LayoutLMv3, Donut |
| **Audio** | Waveform + optional text | Audio → mel spectrogram → encoder | Transcript, embeddings, labels | Whisper, CLAP, Qwen2-Audio |
| **Video** | Sampled frames + text | Frames → per-frame patches → temporal fusion | Text (summaries, answers) | LLaVA-NeXT-Video, VideoLLaVA |

The key insight: despite their differences, **all follow the same pattern**: raw input → processor → tensor representation → model → output.

## 2 — The Processor Pipeline Pattern

Every multimodal model has a **processor** that converts raw inputs into the tensor format the model expects. Understanding the processor is key to understanding what the model "sees":

```
Raw Input (image bytes, audio samples, text)
     │
     ▼
┌─────────────────────┐
│   Processor           │
│   - Resize/normalize  │
│   - Tokenize text     │
│   - Create attention   │
│     masks             │
└─────────┬───────────┘
          │
          ▼
  Tensor dict (input_ids, pixel_values, attention_mask)
          │
          ▼
┌─────────────────────┐
│   Model               │
└─────────┬───────────┘
          │
          ▼
  Output tokens → decoded text
```

In [ ]:
# VLM processor: what happens when you send an image
from PIL import Image
import requests
from io import BytesIO

# Create a simple test image (colored rectangles)
test_img = Image.new("RGB", (320, 240), (70, 130, 180))
from PIL import ImageDraw
draw = ImageDraw.Draw(test_img)
draw.rectangle([20, 20, 150, 120], fill=(220, 50, 50))
draw.rectangle([170, 80, 300, 200], fill=(50, 180, 50))

plt.figure(figsize=(6, 4))
plt.imshow(test_img)
plt.title("Test image for VLM processing")
plt.axis("off")
plt.show()

# Show what the processor produces
messages = [{"role": "user", "content": [
    {"type": "image", "image": test_img},
    {"type": "text", "text": "Describe the shapes and colors in this image."}
]}]

text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text_input], images=[test_img], return_tensors="pt", padding=True)

print("Processor output keys:", list(inputs.keys()))
for k, v in inputs.items():
    if hasattr(v, "shape"):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")

In [ ]:
# Run inference with the VLM
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=200)

# Decode only the generated tokens
generated = output_ids[0][inputs["input_ids"].shape[1]:]
response = processor.tokenizer.decode(generated, skip_special_tokens=True)
print("Model response:")
print(response)

## 3 — Working with VLMs: Chat Templates and Multi-Turn

VLMs use **chat templates** similar to text-only LLMs, but with image placeholders. The message format interleaves text and visual content:

```python
messages = [
    {"role": "user", "content": [
        {"type": "image", "image": pil_image},
        {"type": "text", "text": "What is shown in this image?"},
    ]},
]
```

This format supports multi-turn conversations with images, multi-image inputs, and mixed text+image reasoning.

In [ ]:
# Multi-image comparison
img_a = Image.new("RGB", (200, 200), (220, 50, 50))   # Red square
img_b = Image.new("RGB", (200, 200), (50, 50, 220))   # Blue square

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(img_a); axes[0].set_title("Image A"); axes[0].axis("off")
axes[1].imshow(img_b); axes[1].set_title("Image B"); axes[1].axis("off")
plt.tight_layout()
plt.show()

messages = [{"role": "user", "content": [
    {"type": "image", "image": img_a},
    {"type": "image", "image": img_b},
    {"type": "text", "text": "Compare these two images. What is different?"},
]}]

text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text_input], images=[img_a, img_b], return_tensors="pt", padding=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150)
print(processor.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 4 — Audio Models: Whisper and Beyond

Audio models follow a different processing pipeline:

| Stage | What happens |
|-------|-------------|
| **Resample** | Convert to 16kHz mono |
| **Mel spectrogram** | Time-domain → frequency-domain (80 mel bins) |
| **Chunk** | Split into 30-second windows |
| **Encode** | Transformer encoder produces frame embeddings |
| **Decode** | Autoregressive decoder generates text tokens |

Whisper handles both speech recognition (ASR) and translation in a single model.

In [ ]:
# Audio processing demonstration (without actual audio file)
# Simulate the mel spectrogram that Whisper would see

duration = 10  # seconds
sr = 16000
t = np.linspace(0, duration, sr * duration)

# Simulate speech: fundamental + harmonics + noise
signal = (0.5 * np.sin(2 * np.pi * 150 * t) +
          0.3 * np.sin(2 * np.pi * 300 * t) +
          0.2 * np.sin(2 * np.pi * 450 * t) +
          0.1 * np.random.randn(len(t)))

# Simple mel-like spectrogram visualization
n_fft = 1024
hop = 160  # 10ms at 16kHz
n_frames = len(signal) // hop
n_bins = 80

spec = np.abs(np.random.randn(n_bins, n_frames)) * 0.1
# Add formant-like structure
for center in [15, 35, 55]:
    spec[center-3:center+3, :] += 0.5 + 0.3 * np.sin(np.linspace(0, 20, n_frames))

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
axes[0].plot(t[:3200], signal[:3200], linewidth=0.5)
axes[0].set_title("Waveform (first 200ms)")
axes[0].set_xlabel("Time (s)")

axes[1].imshow(spec, aspect="auto", origin="lower", cmap="magma")
axes[1].set_title(f"Mel Spectrogram ({n_bins} bins x {n_frames} frames)")
axes[1].set_xlabel("Time frames")
axes[1].set_ylabel("Mel bins")
plt.tight_layout()
plt.show()

print(f"Raw audio: {len(signal):,} samples ({duration}s at {sr}Hz)")
print(f"Spectrogram: {n_bins} x {n_frames} = {n_bins * n_frames:,} values")
print(f"After Whisper encoder: ~{n_frames} frame embeddings")

## 5 — The Unified Interface Vision

The field is moving toward **single models that handle all modalities**:

| Model | Text | Image | Audio | Video | Status |
|-------|:----:|:-----:|:-----:|:-----:|--------|
| GPT-4o | ✓ | ✓ | ✓ | ✓ | Proprietary |
| Gemini 1.5 | ✓ | ✓ | ✓ | ✓ | Proprietary |
| Qwen2-VL | ✓ | ✓ | ✗ | ✓ | Open |
| Qwen2-Audio | ✓ | ✗ | ✓ | ✗ | Open |
| LLaVA-NeXT | ✓ | ✓ | ✗ | ✓ | Open |

**Why we still need specialized models**:
- Whisper is more accurate at ASR than any general-purpose model
- ColPali is better at document retrieval than VLMs
- Specialized models are smaller, faster, cheaper
- In production, **accuracy on your task > generality**

In [ ]:
# Model selection matrix: task x constraint -> recommendation
def recommend_model(task, vram_gb=16, needs_speed=False):
    recommendations = {
        "image_qa": [
            ("Qwen2-VL-2B", 2, "Fast, simple QA"),
            ("Qwen2-VL-7B", 5, "Complex reasoning"),
            ("InternVL2-8B", 6, "Best grounding"),
        ],
        "document_search": [
            ("ColPali-3B", 2, "Page-as-image retrieval"),
        ],
        "transcription": [
            ("Whisper-small", 1, "Fast, good quality"),
            ("Whisper-medium", 1, "Better quality"),
            ("Whisper-large-v3", 3, "Best quality"),
        ],
        "audio_understanding": [
            ("CLAP", 1, "Zero-shot classification"),
        ],
        "video_qa": [
            ("Qwen2-VL-7B", 5, "Frame-based understanding"),
        ],
    }

    matches = recommendations.get(task, [])
    valid = [(m, v, d) for m, v, d in matches if v <= vram_gb]
    if needs_speed:
        valid = valid[:1]  # Pick smallest
    return valid if valid else [("None fits", 0, "Reduce constraints")]

for task in ["image_qa", "document_search", "transcription", "audio_understanding", "video_qa"]:
    recs = recommend_model(task, vram_gb=16)
    print(f"\n{task}:")
    for model_name, vram, desc in recs:
        print(f"  {model_name:20s} ({vram}GB) - {desc}")

## 6 — Memory Profiling and Practical Tips

When loading models on Colab Pro, always profile memory usage:

In [ ]:
# Memory profiling
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_mem / 1024**3
    free = total - reserved

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:     {total:.1f} GB")
    print(f"Allocated:      {allocated:.1f} GB")
    print(f"Reserved:       {reserved:.1f} GB")
    print(f"Free:           {free:.1f} GB")
    print()
    print("Tips for Colab Pro T4 (16GB VRAM):")
    print("  - Use 4-bit quantization for models > 3B params")
    print("  - Clear GPU cache between model loads: torch.cuda.empty_cache()")
    print("  - Use del model; torch.cuda.empty_cache() before loading a new model")
    print("  - Mount Google Drive to cache model weights across sessions")

## Exercises

1. **Load and test**: Load Qwen2-VL-3B and test it on 3 different image types (photo, chart, document). Record response quality and latency.

2. **Processor deep-dive**: For any VLM, inspect the processor output shapes for images at 3 different resolutions. How does token count change?

3. **Model comparison**: Pick one task (e.g., chart data extraction). Test with 2 different models. Compare accuracy and speed.

In [ ]:
# Exercise starter code
# Exercise 1: Test on different image types
test_images = {
    "photo": Image.new("RGB", (400, 300), (100, 150, 200)),  # Replace with real images
    "chart": Image.new("RGB", (600, 400), (255, 255, 255)),
    "document": Image.new("RGB", (595, 842), (255, 255, 255)),  # A4 proportions
}

for img_type, img in test_images.items():
    print(f"Image type: {img_type}, size: {img.size}")
    # Add your inference code here

print("Replace placeholder images with real ones for meaningful results.")

## Key Takeaways

| # | Insight |
|---|---------|
| 1 | All multimodal models follow: **raw input → processor → tensors → model → output** |
| 2 | The **processor** determines what the model sees — understand it to debug failures |
| 3 | VLMs use **chat templates** with image/text interleaving for flexible multi-turn interaction |
| 4 | **Specialized models** (Whisper, ColPali) still beat general-purpose models on narrow tasks |
| 5 | Always **profile memory** — know your VRAM budget before loading models |

## What’s Next

→ [03_patches_tokens_spectrograms_and_budgeting.ipynb](03_patches_tokens_spectrograms_and_budgeting.ipynb) — Deep dive into how each modality becomes tokens

## References

- Wang et al., “Qwen2-VL: Enhancing Vision-Language Model’s Perception” (2024)
- Chen et al., “InternVL: Scaling Up Vision Foundation Models” (2024)
- Radford et al., “Robust Speech Recognition via Large-Scale Weak Supervision” (Whisper, 2022)
- Wu et al., “Large-Scale Contrastive Language-Audio Pretraining” (CLAP, 2023)